# Lab 00 · From a prompt to an LLM call

**Time:** 60–90 minutes · **Prerequisites:** basic Python; earlier labs in sequence.

By the end, you can identify the input, model, and output of a call; change a prompt deliberately; and separate a plausible answer from evidence.

## The idea

An LLM generates a response from supplied context. A single call is not an autonomous agent. The application chooses when to call, what context to send, and how to handle failure. This course starts with observable behavior, not hidden reasoning traces.

**Flow:** task → prompt + system instruction → model → text → human check.

**Pacing:** 15 min concept/setup; 20 min worked example; 25 min experiment; 15 min discussion.

## Run this notebook

Use **Runtime → Run all** in Google Colab (CPU runtime is enough), or run locally with Jupyter. No installation or repository clone is needed. The next collapsed cell is the complete shared runtime so this file can travel on its own.

Start in `practice` mode: every response is an explicit, inspectable fixture. It tests Python control flow and failure handling; it is **not evidence of LLM capability**. For the initial live experiment, create a Colab Secret named `FREEINFERENCE_API_KEY`, enable notebook access to that secret, and change `MODE` to `live`. The defaults call `qwen3.6-35b` at `https://freeinference.org/v1`. To use a different compatible gateway, change `BASE_URL`, `MODEL`, `FAMILY` and `SECRET_NAME`; the secret name is configurable. Local Jupyter falls back to a hidden password prompt. Never save a key in a cell or output.

The client sends the same prompts to the configured OpenAI-compatible gateway, bounds calls/output/timeout, limits this notebook to two concurrent network calls, and never silently retries or switches keys. FreeInference requests disable streaming and ask to disable thinking; a model may ignore the thinking setting. The current model page lists the default Qwen model as non-thinking. Other gateways receive no provider-specific thinking field unless configured through `EXTRA_BODY`. A FreeInference cold-start placeholder stops the call with guidance to wait about 120 seconds and retry manually. The teaching team must verify gateway compatibility and available models first; no actual live service calls were made while building this scaffold. If your gateway uses another protocol, ask the TA for an adapter. Token usage is recorded only when reported, cost remains unknown, and practice has no real token usage.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a university credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Ask the university which output-token field it supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use the university HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not family.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model, verified model family, and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or "practice", input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Ask the TA about quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check university access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect university configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Start with FreeInference; change these settings for any compatible gateway. Never paste a key here.
MODE = "practice"       # "live" makes real API calls and may use university quota.
BASE_URL = "https://freeinference.org/v1"  # API base, without /chat/completions
MODEL = "qwen3.6-35b"   # change to the exact identifier for another gateway/model
FAMILY = "Qwen"        # actual model family, verified by the teaching team
SECRET_NAME = "FREEINFERENCE_API_KEY"  # name of your Colab Secret, not the key itself
EXTRA_BODY = {}        # optional gateway-specific fields; cannot override contract/budgets
TOKEN_FIELD = "max_tokens"  # or "max_completion_tokens", according to the gateway

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL, family=FAMILY,
                api_key=secret(SECRET_NAME) if MODE == "live" else "", token_field=TOKEN_FIELD, extra_body=EXTRA_BODY,
                max_calls=24, max_tokens=512, timeout=45)
print("Mode:", MODE, "| All practice outputs are scripted fixtures, not model reasoning.")

## What goes into the LLM call

A live call sends an HTTP POST to the API endpoint. `messages` contains the system instruction and task, `model` selects the configured model, and the token field bounds output length. The runtime adds the authorization header from Colab Secrets; it is deliberately absent from this printable example. This cell builds a payload only and makes no network call.

In [ ]:
example_payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Compare architecture alternatives; state uncertainty."},
        {"role": "user", "content": "What information is missing before choosing an event-ingestion design?"},
    ],
    TOKEN_FIELD: 128,
}
if urllib.parse.urlsplit(BASE_URL).hostname == "freeinference.org":
    example_payload.update({"stream": False, "thinking": {"type": "disabled"}})
print("POST", BASE_URL.rstrip("/") + "/chat/completions")
print(json.dumps(example_payload, indent=2))  # no key or authorization header
assert "Authorization" not in example_payload and "api_key" not in example_payload

## A first call

Ask for an architecture recommendation with two alternatives. The fixture is intentionally incomplete: it names a choice without measurements. Live output may differ. Do not treat model confidence as evidence.

In [ ]:
prompt = "A small team needs event ingestion. Compare a lightweight service and a durable event log in two sentences. Name one missing requirement."
answer = client.ask(prompt, system="Give a concise architecture comparison. State uncertainty.",
    practice="A lightweight service reduces operational effort; a durable log adds replay at extra complexity. Missing requirement: how long must events be retained?")
print(answer)
assert isinstance(answer, str) and answer.strip(), "An empty response is a failed call, not a recommendation."

## Make one controlled prompt change

Add a hard requirement, keep the rest of the prompt fixed, and compare answers. This is a small qualitative experiment, not a reliability benchmark.

In [ ]:
constraint = "Hard requirement: events must be replayable for seven days."
constrained_answer = client.ask(prompt + " " + constraint,
    system="Give a concise architecture comparison. State uncertainty.",
    practice="The durable log better matches the explicit replay requirement in this exercise. We still need retention capacity and operations estimates.")
print(constrained_answer)
print(json.dumps(summarize_trace([client]), indent=2))
assert len(client.trace) == 2
assert all("api_key" not in row and "prompt" not in row for row in client.trace)

**Expected observation:** the second fixture responds to the new constraint. With a live model, inspect whether it actually uses the requirement and whether it invents throughput numbers. A fluent answer alone does not establish correctness. `None` for token counts or cost means unknown; summed call latency is not an end-to-end benchmark.

## Your experiment

Change `constraint` to require on-premise operation or a strict latency target. Before running, write which evidence you would need to justify a recommendation. Try a vague prompt and a specific prompt with the same model. Record a counterexample where an answer sounds certain but does not support its claim. Set a fresh practice `Client(max_calls=1)`, call it twice, and confirm the second call stops at the budget.

## Bridge to the architect project

This becomes the **single-call baseline** in Experiment A. Save the exact prompt, model identifier, input context and output; the later council must be compared with each participating model alone.

## Submit for this lab

Save your modified notebook with secrets removed and include: (1) the change you made, (2) one successful and one failing example, (3) what the assertions establish, (4) an observation from a real model if university access is enabled, and (5) the call/token/latency trace. Explain any unknown metrics instead of inventing them.

## Read further

- [FreeInference quickstart](https://doc.freeinference.org/quickstart)
- [FreeInference model catalog](https://doc.freeinference.org/models)
- [OpenAI-compatible Chat Completions request/response schema](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents)